### 1. Testing method

NB: Already went through paraphrasing prior to this testing the method but assume you don't want any more IPYNBs than necessary so left that one out. First section of this file tests the method works as expected. Second half actually runs the translation of the 2000 row dataset.

______________

In [47]:
from openai import OpenAI
from bert_score import score
import pandas as pd
import numpy as np
import getpass
import time
import os

In [48]:
# use getpass to pass api key over so as not to inclue here
API_KEY = getpass.getpass("Enter OpenAI API key: ")

# create client
client = OpenAI(api_key = API_KEY)

print("Open AI client created.")

Enter OpenAI API key:  ········


Open AI client created.


In [19]:
# set up csv, take copy of first 20 rows for initial testing

csv_file = "final_human_dataset_copy.csv"
df = pd.read_csv(csv_file)
test = df.head(20).copy()
print("Test text: ", len(test))

Test text:  20


Now need to set up the prompts that will be passed to Open AI for paraphrasing. Seperate prompts for light paraphrase, heavy paraphrase, translation to French and translation from French back to English.

In [49]:
light_para_prompt = """

Paraphrase the following text lightly.

Make limited changes to the original wording, phrasing and sentence structure.
Replace some words and phrases with alternatives.
Make minor grammatical changes.

Preserve the original meaning, factual information, level of detail, structure and ideas as closely as possible. 
The result should be a paraphrase that remains very close to the original text.
Do not add new information, remove information, or summarise the text.
Keep the length similar to the original.

Return only the paraphrased text.


TEXT: {text}

"""

In [50]:
heavy_para_prompt = """

Paraphrase the following text heavily.

Make substantial changes to the original wording, phrasing and sentence structure.
Make major grammatical changes, and use substantially different vocabulary.
Reorganise sentences and vary how ideas are expressed where appropriate.


Preserve the original meaning, factual information, level of detail, and ideas as closely as possible.
The result should be a paraphrase that is substantially different in wording, structure and order to 
the original text whilst preserving its meaning.
Do not add new information, remove information, or summarise the text.
Keep the length similar to the original.

Return only the paraphrased text.


TEXT: {text}

"""

In [51]:
trans_french_prompt = """

Translate the following English text into French.

Create a natural and fluent translation.
Preserve the original meaning, factual information, level of detail, and ideas as closely as possible.
Preserve any names, dates, numbers and technical information.

Do not add new information, remove information, or summarise the text.
Keep the length similar to the original.

Return only the translated French text.


TEXT: {text}

"""

In [52]:
trans_english_prompt = """

Translate the following French text into English.

Create a natural and fluent translation.
Preserve the original meaning, factual information, level of detail, and ideas as closely as possible.
Preserve any names, dates, numbers and technical information.

Do not add new information, remove information, or summarise the text.
Keep the length similar to the original.

Return only the translated English text.


TEXT: {text}

"""

Do a test on one sample...

In [53]:
# function to send prompt 
def send_prompt(prompt, model):

    # send request to open AI
    response = client.responses.create(
        # specify model
        model = model,
        # send prompt
        input = prompt
    )

    # get response, trim whitespace
    return response.output_text.strip()


In [24]:
# get text from first row of test data
text = test.iloc[5]["text"]

# send light paraphrasing prmopt to llm
light = send_prompt(
    light_para_prompt.format(text = text),
    # specify luna (cheaper / lesser) model
    "gpt-5.6-luna"
)

# same but for heavy
heavy = send_prompt(
    heavy_para_prompt.format(text = text),
    "gpt-5.6-luna"
)

# run translation to french
french = send_prompt(
    trans_french_prompt.format(text = text),
    "gpt-5.6-luna"
)

# translate that french back to english
english = send_prompt(
    # input is now the french translation
    trans_english_prompt.format(text = french),
    "gpt-5.6-luna"
)

print("Original\n")
print(text)

print("\n\nLight\n")
print(light)

print("\n\nHeavy\n")
print(heavy)

print("\n\nFrench\n")
print(french)

print("\n\nTranslated back to English\n")
print(english)


Original

Severe Tropical Cyclone Xavier was a strong pre-season cyclone that formed on October 20, 2006, to the north of the Santa Cruz Islands. Cyclone Xavier was also the strongest storm of the season. During the next day it rapidly developed and was designated as Tropical Depression 01F later that day, before being designated as Tropical Cyclone Xavier while over Santa Cruz early on October 22. Later on October 22, Xavier intensified into a category 3 severe tropical cyclone on the Australian Tropical Cyclone Intensity Scale, before passing over Tikopia early the next day. Xavier then reached its peak wind speeds of 175 km/h (110 mph 10-min), which made it a Category 4 cyclone early on October 24. Xavier stayed at its peak wind speeds until early the next day when it started to rapidly weaken, becoming a depression early on October 26. The remnants of Xavier persisted until October 28 before dissipating. There were no casualties reported as a result of the storm and little impact r

Appears to be working as intended. Now need to look at Bert score and edit distance to confirm this across the 20 test samples.

In [27]:
# two possible models to decide between at this stage as discussed in report

models = ["gpt-5.6-terra", "gpt-5.6-luna"]

In [31]:
# now want to loop through 20 texts in test df, and for each carry out light, heavy and back translation (eng > fre > eng)
# and also do this using each of the two possible models

# initialise empty results list
results = []

# loop through test df
for i, (_, row) in enumerate (test.iterrows(), start = 1):

    print(f"\n====== Document {i}/20 ======")
    # get original text and id from test df
    document_id = row["document_id"]
    original_text = row["text"]
    print("Document: ", document_id)

    # now for each of the two possible models, terra and luna
    for model in models:

        print("Model: ", model)

        try:

            # light paraphrase
            # add original text from df into light paraphrase prompt
            light = send_prompt(
                light_para_prompt.format(text = original_text),
                model
            )

            # heavy paraphrase
            # add original text from df into heavy paraphrase prompt
            heavy = send_prompt(
                heavy_para_prompt.format(text = original_text),
                model
            )

            # eng to fre translation
            # add original text from df into translation prompt
            french = send_prompt(
                trans_french_prompt.format(text = original_text),
                model
            )

            # fre to eng translation
            # add french text into translation prompt
            english = send_prompt(
                trans_english_prompt.format(text = french),
                model
            )

            # store output
            results.append({
                "document_id": document_id,
                "source": row["source"],
                "topic": row["topic"],
                "original_text": original_text,
                "model": model,
                "light_paraphrase": light,
                "heavy_paraphrase": heavy,
                "french_translation": french,
                "translation_paraphrase": english
            })

            print(f"Completed: {document_id}, {model}")

        # catch any errors instead of stopping loop
        except Exception as e:

            print("Error:", e)

            # still want to store output to find issue later if necessary
            results.append({
                "document_id": document_id,
                "source": row["source"],
                "topic": row["topic"],
                "original_text": original_text,             
                "model": model,
                "light_paraphrase": None,
                "heavy_paraphrase": None,
                "french_translation": None,
                "translation_paraphrase": None
            })

            print("Failed: ", document_id)

        # small half second pause between models to avoid overuse error
        time.sleep(0.5)


====== Document 1/20 ======
Document:  WIKI_0122
Model:  gpt-5.6-terra
Completed: WIKI_0122, gpt-5.6-terra
Model:  gpt-5.6-luna
Completed: WIKI_0122, gpt-5.6-luna

====== Document 2/20 ======
Document:  BBC_0281
Model:  gpt-5.6-terra
Completed: BBC_0281, gpt-5.6-terra
Model:  gpt-5.6-luna
Completed: BBC_0281, gpt-5.6-luna

====== Document 3/20 ======
Document:  WIKI_0313
Model:  gpt-5.6-terra
Completed: WIKI_0313, gpt-5.6-terra
Model:  gpt-5.6-luna
Completed: WIKI_0313, gpt-5.6-luna

====== Document 4/20 ======
Document:  AMZ_0214
Model:  gpt-5.6-terra
Completed: AMZ_0214, gpt-5.6-terra
Model:  gpt-5.6-luna
Completed: AMZ_0214, gpt-5.6-luna

====== Document 5/20 ======
Document:  BBC_0278
Model:  gpt-5.6-terra
Completed: BBC_0278, gpt-5.6-terra
Model:  gpt-5.6-luna
Completed: BBC_0278, gpt-5.6-luna

====== Document 6/20 ======
Document:  WIKI_0204
Model:  gpt-5.6-terra
Completed: WIKI_0204, gpt-5.6-terra
Model:  gpt-5.6-luna
Completed: WIKI_0204, gpt-5.6-luna

====== Document 7/20 ===

In [35]:
para_df = pd.DataFrame(results)

In [36]:
para_df

,document_id,source,topic,original_text,model,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase
0,WIKI_0122,Wikipedia,History,ʿIzz ad-Dīn ibn Abd al-Qāder ibn Mustafā ibn Y...,gpt-5.6-terra,ʿIzz ad-Dīn ibn Abd al-Qāder ibn Mustafā ibn Y...,ʿIzz ad-Dīn ibn Abd al-Qāder ibn Mustafā ibn Y...,ʿIzz ad-Dīn ibn Abd al-Qāder ibn Mustafā ibn Y...,ʿIzz ad-Dīn ibn Abd al-Qāder ibn Mustafā ibn Y...
1,WIKI_0122,Wikipedia,History,ʿIzz ad-Dīn ibn Abd al-Qāder ibn Mustafā ibn Y...,gpt-5.6-luna,ʿIzz ad-Dīn ibn Abd al-Qāder ibn Mustafā ibn Y...,ʿIzz ad-Dīn ibn Abd al-Qāder ibn Mustafā ibn Y...,ʿIzz ad-Dīn ibn Abd al-Qāder ibn Mustafā ibn Y...,ʿIzz ad-Dīn ibn Abd al-Qāder ibn Mustafā ibn Y...
2,BBC_0281,BBC,General,"Under international law, South Africa had a du...",gpt-5.6-terra,"Under international law, South Africa was obli...",A judge said South Africa was legally obliged ...,"En vertu du droit international, l’Afrique du ...","Under international law, South Africa had a du..."
3,BBC_0281,BBC,General,"Under international law, South Africa had a du...",gpt-5.6-luna,"Under international law, South Africa was obli...",Judge Mohammed Navsa ruled that international ...,"En vertu du droit international, l’Afrique du ...","Under international law, South Africa had a du..."
4,WIKI_0313,Wikipedia,Sports and recreation,"Washington Square, also known as Washington Sq...",gpt-5.6-terra,"Washington Square, also called Washington Squa...","Washington Square, commonly called Washington ...","Washington Square, également connu sous le nom...","Washington Square, also known as Washington Sq..."
5,WIKI_0313,Wikipedia,Sports and recreation,"Washington Square, also known as Washington Sq...",gpt-5.6-luna,"Washington Square, also called Washington Squa...","Washington Square, commonly called Washington ...","Washington Square, également connu sous le nom...","Washington Square, also known as Washington Sq..."
6,AMZ_0214,Amazon reviews,Food reviews,We love Happy Tots food. My 12 month old has t...,gpt-5.6-terra,We love Happy Tots food. My 12-month-old has t...,We are big fans of Happy Tots products. My 12-...,Nous adorons les aliments Happy Tots. Mon enfa...,We love Happy Tots foods. My 12-month-old has ...
7,AMZ_0214,Amazon reviews,Food reviews,We love Happy Tots food. My 12 month old has t...,gpt-5.6-luna,We really like Happy Tots food. My 12-month-ol...,We’re very pleased with Happy Tots food. My 12...,Nous adorons les aliments Happy Tots. Mon bébé...,We love Happy Tots foods. My 12-month-old baby...
8,BBC_0278,BBC,General,Chairman Paulo Sergio Pinheiro urged the inter...,gpt-5.6-terra,Chairman Paulo Sergio Pinheiro called on the i...,"Paulo Sergio Pinheiro, the commission’s chairm...",Le président Paulo Sergio Pinheiro a exhorté l...,Chairman Paulo Sergio Pinheiro urged the inter...
9,BBC_0278,BBC,General,Chairman Paulo Sergio Pinheiro urged the inter...,gpt-5.6-luna,Chairman Paulo Sergio Pinheiro called on the i...,"Paulo Sergio Pinheiro, the commission’s chair,...",Le président Paulo Sergio Pinheiro a exhorté l...,President Paulo Sergio Pinheiro urged the inte...


In [42]:
print(para_df.iloc[4]["original_text"])
print("\n")
print(para_df.iloc[4]["heavy_paraphrase"])

Washington Square, also known as Washington Square Park, is a public park and registered historic landmark in Chicago, Illinois. Historically known by its nickname Bughouse Square referring to mental health facilities, it was the most celebrated open air free-speech center in the country. It is located across Walton Street from Newberry Library at 901 N. Clark Street in the Near North Side community area of Chicago. Washington Square Park is Chicago's oldest existing small park. It is one of four Chicago Park District parks named after persons surnamed Washington (the others being Washington Park, Harold Washington Park, and Dinah Washington Park). It was added to the National Register of Historic Places on May 20, 1991. On September 4, 1842, the city received a 3-acre (12,000 m2) parcel that was donated by the members of the American Land Company for use as a public park. The property had once been a cow path with a well for farmers to water their cattle. The donors stipulated the nam

Edit distance measures surface-level changes - characters/sequqnces etc. BERT score  measures semantic similarity - the meaning, regardless of the words. 
Edit distance = 0 - 1 @ 0 is identical, 1 different
BERT score = 0 - 1 @ 0 is different, 1 similar meaning
So we expect BERT score to decrease from light to heavy para, and edit to increase. We will wait to see with translation, but expect closer to heavy para stats.

In [3]:
# need to calculate normalized edit distance and BERT scores

# this block calculates edit distance
# loop through the three paraphrase types
for para_type in ["light", "heavy", "translation"]:

    # name of para col.
    paraphrase_col = f"{para_type}_paraphrase"
    # name of score col.
    score_col = f"edit_{para_type}"

    # calculate scores
    para_df[score_col] = para_df.apply(

        # loop row by row to calculate
        lambda row: (

            # calculate edit distance
            distance(
                str(row["original_text"]),
                str(row[paraphrase_col])
            )

            # divide by the longer text (original vs para)
            / max(
                len(str(row["original_text"])),
                len(str(row[paraphrase_col]))
            )
        ),
        axis = 1
    )


# this block calcualtes BERT scores
# get all the original texts
original_texts = para_df["original_text"].tolist()

# loop through the three paraphrase types
for para_type in ["light", "heavy", "translation"]:

    # name of para col.
    paraphrase_col = f"{para_type}_paraphrase"
    # name of score col.
    score_col = f"BERT_{para_type}"

    # get all paraphrases as list
    paraphrases = para_df[paraphrase_col].tolist()

    # get BERT score
    P, R, F1 = score(
        paraphrases,
        original_texts,
        lang = "en",
    )

    # F1 BERT score is the key metric we want, save it for each text
    para_df[score_col] = F1.tolist()

# now create a new DF to inspect the new scores
scores = para_df[
    [
        "document_id",
        "model",
        "BERT_light",
        "BERT_heavy",
        "BERT_translation",
        "edit_light",
        "edit_heavy",
        "edit_translation"
    ]
]

scores.head(20)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,document_id,model,BERT_light,BERT_heavy,BERT_translation,edit_light,edit_heavy,edit_translation
0,WIKI_0122,gpt-5.6-terra,0.984579,0.958413,0.981265,0.148148,0.409056,0.139124
1,WIKI_0122,gpt-5.6-luna,0.979677,0.952895,0.972157,0.140893,0.443741,0.190494
2,BBC_0281,gpt-5.6-terra,0.984621,0.943634,0.966811,0.131739,0.389551,0.186259
3,BBC_0281,gpt-5.6-luna,0.966258,0.926601,0.945694,0.147896,0.482012,0.216602
4,WIKI_0313,gpt-5.6-terra,0.971246,0.933661,0.965428,0.197469,0.474788,0.169062
5,WIKI_0313,gpt-5.6-luna,0.976054,0.923002,0.951096,0.137958,0.482304,0.235905
6,AMZ_0214,gpt-5.6-terra,0.981885,0.941472,0.969781,0.078431,0.330402,0.127336
7,AMZ_0214,gpt-5.6-luna,0.983141,0.932601,0.933745,0.119965,0.390390,0.238273
8,BBC_0278,gpt-5.6-terra,0.983235,0.922784,0.957342,0.129776,0.382724,0.222186
9,BBC_0278,gpt-5.6-luna,0.979991,0.919553,0.941283,0.158746,0.497966,0.248489


NB: As mentioned at top I actually tested paraphrasing in another IPYNB prior to this and this is where I first tested edit / bert scores. Data in report is actually from that IPYNB so the values below might differ slightly.

Looks to be working as intended, lets check means

In [5]:
means = para_df.groupby("model")[
    [
        "BERT_light",
        "BERT_heavy",
        "BERT_translation",
        "edit_light",
        "edit_heavy",
        "edit_translation"
    ]
].mean()

means

,BERT_light,BERT_heavy,BERT_translation,edit_light,edit_heavy,edit_translation
model,,,,,,
gpt-5.6-luna,0.970486,0.914807,0.942565,0.151462,0.513899,0.248215
gpt-5.6-terra,0.978727,0.924639,0.956488,0.138422,0.490558,0.206990


____________

### 2. Create paraphrase dataset

I now move onto paraphrasing the whole 2000 pieces of human written text using the Luna model.

________

In [8]:
# initially only took first 20 rows as a test, now want copy of whole thing

csv_file = "final_human_dataset_copy.csv"
df = pd.read_csv(csv_file)
main = df.copy()

In [9]:
# given the time taken to run the api calls for just the initial 20 texts i forsee this full dataset to take a long time, up to 24 hours.
# dont want it to crash out and if it does, or if my api credit runs out i dont want to have to start from the start again etc. hence
# why this is much more robust than the above code.

In [10]:
# set new output file to contain all original text and info and new paraphrases
output_file = "full_paraphrase_dataset.csv"

# have now decided on luna model
model = "gpt-5.6-luna"


# if this does crash, I want to be able to re-run this cell which should check for existing
# data to determine where to carry on from, rather than re-doing old work
if os.path.exists(output_file):

    # read output file
    para_df = pd.read_csv(output_file)
    # convert df to list of dicts
    results = para_df.to_dict("records")
    # find number completed already
    print("Existing results found: ", len(results))

else:

    # if there were none existing this is first time therefore initialise list
    results = []
    print("Starting from beginning.")



# have established where we are startign from, now need to get the ids of any already complete
completed = {
    row["document_id"]
    for row in results
    if pd.notna(row.get("light_paraphrase"))
    and pd.notna(row.get("heavy_paraphrase"))
    and pd.notna(row.get("translation_paraphrase"))
}
print("Documents completed: ", len(completed))


############################################
# Begin looping through dataset
############################################

# now want to loop through whole dataset, initially skipping any rows that have
# a document ID within completed, established above
for i, (_, row) in enumerate(main.iterrows(), start = 1):

    # get id and text data from csv
    document_id = row["document_id"]
    original_text = row["text"]

    # now look for those ids in completed to see what can be skipped
    if document_id in completed:

        print(
            f"[{i}/{len(main)} ]"
            f"{document_id} already completed"
        )

        # if id existed move onto next and do nothing
        continue

    # otherwise we are now beginning to paraphrase those that havne't been done already
    print(f"\n====== Document {i}/{len(main)} ======")
    print("Document: ", document_id)

    
    ############################################
    # Now we move onto the paraphrasing requests
    ############################################

    try:

        # light paraphrase
        print("Light paraphrase...")
        # add original text from df into light paraphrase prompt
        light = send_prompt(
            light_para_prompt.format(text = original_text),
            model
        )

        # Heavy paraphrase
        print("Heavy paraphrase...")
        # add original text from df into heavy paraphrase prompt
        heavy = send_prompt(
            heavy_para_prompt.format(text = original_text),
            model
        )

        # eng to fre translation
        print("Eng > Fre translation...")
        # add original text from df into translation prompt
        french = send_prompt(
            trans_french_prompt.format(text = original_text),
            model
        )

        # fre to eng translation
        print("Fre > Eng translation...")
        # add french text into translation prompt
        english = send_prompt(
            trans_english_prompt.format(text = french),
            model
        )

        # store output
        results.append({
            "document_id": document_id,
            "source": row["source"],
            "topic": row["topic"],
            "model": model,
            "original_text": original_text,
            "light_paraphrase": light,
            "heavy_paraphrase": heavy,
            "french_translation": french,
            "translation_paraphrase": english
        })

        # also add doc id to completed
        completed.add(document_id)
        print("Document complete")


    ############################################
    # Handle API request errors
    ############################################

    # want to state what the error was and still add the row to the document 
    # so that it is logged and we can continue
    except Exception as e:

        print("Error:", e)

        # still want to store output to find issue later if necessary
        results.append({
            "document_id": document_id,
            "source": row["source"],
            "topic": row["topic"],
            "model": model,
            "original_text": original_text,             
            "light_paraphrase": None,
            "heavy_paraphrase": None,
            "french_translation": None,
            "translation_paraphrase": None
        })

        print("Failed: ", document_id)


    ############################################
    # Save
    ############################################

    # want to save to the csv after each document rather than waiting 
    # until very end in case it crashes
    para_df = pd.DataFrame(results)
    para_df.to_csv(output_file, index = False)
    print("Results saved")
    # small half second pause between models to avoid overuse error
    time.sleep(0.5)

Existing results found:  1907
Documents completed:  1904
[1/2000 ]WIKI_0122 already completed
[2/2000 ]BBC_0281 already completed
[3/2000 ]WIKI_0313 already completed
[4/2000 ]AMZ_0214 already completed
[5/2000 ]BBC_0278 already completed
[6/2000 ]WIKI_0204 already completed
[7/2000 ]WIKI_0359 already completed
[8/2000 ]GUT_0281 already completed
[9/2000 ]RED_0322 already completed
[10/2000 ]BBC_0192 already completed
[11/2000 ]BBC_0077 already completed
[12/2000 ]WIKI_0353 already completed
[13/2000 ]RED_0142 already completed
[14/2000 ]GUT_0078 already completed
[15/2000 ]GUT_0191 already completed
[16/2000 ]AMZ_0089 already completed
[17/2000 ]RED_0084 already completed
[18/2000 ]RED_0228 already completed
[19/2000 ]GUT_0176 already completed
[20/2000 ]AMZ_0165 already completed
[21/2000 ]GUT_0116 already completed
[22/2000 ]BBC_0320 already completed
[23/2000 ]WIKI_0389 already completed
[24/2000 ]WIKI_0012 already completed
[25/2000 ]GUT_0220 already completed
[26/2000 ]GUT_0395 a

In [11]:
# had to run multiple times so good that it was made so robust, now need to verify that we actually have what we expect and can mvoe on

In [9]:
# re-loading in here

In [10]:
csv_file = "full_paraphrase_dataset.csv"
para_df = pd.read_csv(csv_file)

In [11]:
len(para_df)

2003

In [12]:
len(para_df.columns)

9

2003 is three more than expected so must assume duplicates?

In [13]:
para_df["document_id"].duplicated().sum()

3

In [14]:
duplicates = para_df[para_df["document_id"].duplicated(keep = False)].sort_values("document_id")

In [15]:
duplicates

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase
658,AMZ_0349,Amazon reviews,Food reviews,gpt-5.6-luna,I have an almost insatiable sweet tooth. Left ...,NaN,NaN,NaN,NaN
1908,AMZ_0349,Amazon reviews,Food reviews,gpt-5.6-luna,I have an almost insatiable sweet tooth. Left ...,I have an almost impossible-to-satisfy sweet t...,I have an extremely persistent craving for swe...,J’ai une dent sucrée presque insatiable. Si on...,I have an almost insatiable sweet tooth. If le...
659,GUT_0222,Gutenberg,English fiction,gpt-5.6-luna,"CHAPTER III. “ARCADES AMBO,” BUT FLOURISHING N...",NaN,NaN,NaN,NaN
1909,GUT_0222,Gutenberg,English fiction,gpt-5.6-luna,"CHAPTER III. “ARCADES AMBO,” BUT FLOURISHING N...","CHAPTER III. “ARCADES AMBO,” BUT PROSPERING AL...","CHAPTER III. “ARCADES AMBO,” YET PROSPERING AL...","CHAPITRE III. « ARCADES AMBO », MAIS PROSPÈRE ...","CHAPTER III. “ARCADES AMBO,” BUT PROSPEROUS NE..."
657,RED_0039,Reddit,relationship_advice,gpt-5.6-luna,"Once again, I did not insist that he went. He ...",NaN,NaN,NaN,NaN
1907,RED_0039,Reddit,relationship_advice,gpt-5.6-luna,"Once again, I did not insist that he went. He ...","Once again, I did not demand that he go. He vo...",I never demanded that he come along; he volunt...,"Encore une fois, je n’ai pas insisté pour qu’i...","Once again, I didn’t insist that he come. He o..."


Ahh, these were three that didn't work for some reason on the first pass and were picked up and re-run later but it kept the old ones as NaNs, so need to remove those

In [16]:
para_df = para_df.drop_duplicates(subset = "document_id", keep = "last").reset_index(drop = True)

In [17]:
para_df["document_id"].duplicated().sum()

0

In [18]:
para_df.isna().sum()

document_id               0
source                    0
topic                     0
model                     0
original_text             0
light_paraphrase          0
heavy_paraphrase          0
french_translation        0
translation_paraphrase    0
dtype: int64

Now no duplicates and no NaNs. Should double check word counts to ensure we retained the 350-450 range after paraphrasing.

In [21]:
# split string in each cell in the original text row to words to calculate
# word count, do for every row in column...
para_df["original_word_count"] = (
    para_df["original_text"]
    .fillna("")
    .str.split()
    .str.len()
)

# and do for all 5 text columns...
para_df["light_word_count"] = (
    para_df["light_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

para_df["heavy_word_count"] = (
    para_df["heavy_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

para_df["french_word_count"] = (
    para_df["french_translation"]
    .fillna("")
    .str.split()
    .str.len()
)
para_df["translation_word_count"] = (
    para_df["translation_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

print(para_df[
    [
        "original_word_count",
        "light_word_count",
        "heavy_word_count",
        "french_word_count",
        "translation_word_count"
    ]
].describe())

       original_word_count  light_word_count  heavy_word_count  \
count          2000.000000       2000.000000       2000.000000   
mean            398.800000        390.056500        404.742500   
std              23.693748         28.240575         35.802054   
min             311.000000         17.000000        109.000000   
25%             381.000000        374.000000        381.000000   
50%             404.000000        395.000000        407.000000   
75%             414.000000        408.000000        428.000000   
max             451.000000        455.000000        525.000000   

       french_word_count  translation_word_count  
count        2000.000000             2000.000000  
mean          458.612500              408.132000  
std            38.413984               28.448044  
min           343.000000              265.000000  
25%           432.000000              390.000000  
50%           458.000000              410.000000  
75%           485.000000              426.000000

Some rather large ones I am happy to ignore but there is a 17 word parpahrase which is not acceptable, along with minimums under 300 so need to find and investigate.

In [22]:
# new df containing rows where word count for one of the paraphrases / translations is < 300
short_text = para_df[
    (para_df["light_paraphrase"].fillna("").str.split().str.len() < 300) |
    (para_df["heavy_paraphrase"].fillna("").str.split().str.len() < 300) |
    (para_df["french_translation"].fillna("").str.split().str.len() < 300) |
    (para_df["translation_paraphrase"].fillna("").str.split().str.len() < 300)
    ]

short_text[
    [
        "document_id",
        "original_text",
        "light_paraphrase",
        "heavy_paraphrase",
        "translation_paraphrase"
    ]
]        

,document_id,original_text,light_paraphrase,heavy_paraphrase,translation_paraphrase
162,BBC_0375,"The 15-year-old, who is currently staying with...","The 15-year-old, who is currently staying with...",The 15-year-old is currently being sheltered b...,"The 15-year-old girl, who is currently staying..."
277,RED_0127,"For one, the thing about not getting to choose...","For one thing, the idea that you do not get to...","To begin with, the claim that communism means ...","First of all, the idea that you cannot choose ..."
299,BBC_0309,Anatoly Pakhomov said homosexuals were welcome...,Anatoly Pakhomov said homosexuals were welcome...,Sochi mayor Anatoly Pakhomov said gay people w...,Anatoli Pakhomov said that gay people would be...
430,AMZ_0156,Dogswell jerky treat come China. They are not ...,Dogswell jerky treats come from China. They ar...,"Dogswell jerky treats are made in China, and t...",Dogswell dried treats come from China. They ar...
690,RED_0111,"This is my first answer to an ELI5, so stick w...","This is my first answer to an ELI5, so bear wi...",This is my first attempt at answering an ELI5 ...,"This is my first response to an ELI5, so pleas..."
738,BBC_0080,Officer Jason Van Dyke is charged with the fir...,Officer Jason Van Dyke has been charged with t...,Chicago police officer Jason Van Dyke faces a ...,Officer Jason Van Dyke is accused of the first...
975,BBC_0344,Divers from the FBI began searching the waters...,FBI divers began combing the waters of Seccomb...,FBI divers were deployed to Seccombe Lake Park...,FBI divers have begun searching the waters of ...
1437,BBC_0148,Researchers from the Victor Chang Institute in...,Researchers from Sydney’s Victor Chang Institu...,Scientists at Sydney’s Victor Chang Institute ...,Researchers at the Victor Chang Institute in S...
1462,WIKI_0010,South Shore Line Airport Realignment is the na...,South Shore Line Airport Realignment is the na...,The South Shore Line Airport Realignment refer...,The South Shore Line Airport Redevelopment is ...
1511,AMZ_0394,Let me say this first: our family of 6 keeps a...,Let me say this first: our family of 6 follows...,"To begin with, our household of six follows a ...",Let me say this first: our family of six follo...


Can now inspect some of these manually.

In [32]:
para_df[para_df["document_id"] == "BBC_0375"]

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
162,BBC_0375,BBC,General,gpt-5.6-luna,"The 15-year-old, who is currently staying with...","The 15-year-old, who is currently staying with...",The 15-year-old is currently being sheltered b...,"L’adolescente de 15 ans, qui séjourne actuelle...","The 15-year-old girl, who is currently staying...",357,249,379,432,359


This one is low on the light paraphrase so lets check.

In [23]:
row = para_df[para_df["document_id"] == "BBC_0375"].iloc[0]
print(row["light_paraphrase"])

The 15-year-old, who is currently staying with relatives, has been targeted in an online and in-person harassment campaign. People on Facebook have threatened that “they are going to get” her daughter, the 32-year-old woman told US media. The BBC is not naming the mother to protect the girl’s privacy. About 40 people were reportedly watching the livestream, but no one notified police about the incident. The girl has been too afraid to return to the Lawndale neighbourhood where she lives. Her mother says local children have been ringing her doorbell searching for the girl. “This is simply disturbing, and to think the kids find it funny,” she said, adding that she now wants to leave the crime-ridden west Chicago neighbourhood. “I can’t stay here,” she said on Wednesday after being reunited with her daughter.

Chicago police located the girl one day after she disappeared and contacted Facebook to get the video taken down. The girl has now been released from hospital, but has not yet gone 

Can see that the text cut out in the middle of the light para for BBC_0375, lets check others.

In [24]:
para_df[para_df["document_id"] == "RED_0127"]

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
277,RED_0127,Reddit,explainlikeimfive,gpt-5.6-luna,"For one, the thing about not getting to choose...","For one thing, the idea that you do not get to...","To begin with, the claim that communism means ...","Tout d’abord, l’idée selon laquelle on ne peut...","First of all, the idea that you cannot choose ...",407,324,421,416,265


In [25]:
# check the translation 
row = para_df[para_df["document_id"] == "RED_0127"].iloc[0]
print(row["translation_paraphrase"])

First of all, the idea that you cannot choose what you do is simply false. There are many types of communism. Stalinist communism is not the kind that most communists would support, nor is Maoist communism, even though some people believe in them. China used to be communist, but it is now practically as capitalist as any other country, so it is a poor example. The most important idea is that the government owns the means of production. Today, let’s say you go to work in a factory, and you make stuffed animals. You might earn $8 an hour, while the factory owners probably make $16 an hour from your labor. Communists do not like this. They find it absurd that simply owning the factory allows you to profit from everyone else’s work. Instead, they want the government to own the factory. Now, no one makes a profit, but the government provides for all your needs. It will distribute the teddy bears and make sure that everyone has a job to do. Under Stalinism or Maoism, many people could not ch

Can see it also cut out in the middle of a sentence.

In [26]:
para_df[para_df["document_id"] == "BBC_0309"]

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
299,BBC_0309,BBC,General,gpt-5.6-luna,Anatoly Pakhomov said homosexuals were welcome...,Anatoly Pakhomov said homosexuals were welcome...,Sochi mayor Anatoly Pakhomov said gay people w...,Anatoli Pakhomov a déclaré que les homosexuels...,Anatoli Pakhomov said that gay people would be...,427,17,439,507,440


In [27]:
row = para_df[para_df["document_id"] == "BBC_0309"].iloc[0]
print(row["light_paraphrase"])

Anatoly Pakhomov said homosexuals were welcome at the Games, provided they “respect Russian law” and “do not


Same again. Its clear that some paraphrases / translations were just cut out midway through for some reason, possibly an issue with the API. Lets check a few more to be sure.

In [28]:
para_df[para_df["document_id"] == "AMZ_0394"]

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
1511,AMZ_0394,Amazon reviews,Food reviews,gpt-5.6-luna,Let me say this first: our family of 6 keeps a...,Let me say this first: our family of 6 follows...,"To begin with, our household of six follows a ...",Permettez-moi de dire ceci en premier : notre ...,Let me say this first: our family of six follo...,311,289,321,398,320


It appears a word count of 311 for the original slipped through here and so a light paraphrase of 289 is acceptable. No need to delete this one as it still contains appropriate content.

In [30]:
para_df[para_df["document_id"] == "WIKI_0010"]

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
1462,WIKI_0010,Wikipedia,Engineering and technology,gpt-5.6-luna,South Shore Line Airport Realignment is the na...,South Shore Line Airport Realignment is the na...,The South Shore Line Airport Realignment refer...,Le réaménagement de l’aéroport de la South Sho...,The South Shore Line Airport Redevelopment is ...,401,363,279,435,369


In [31]:
row = para_df[para_df["document_id"] == "WIKI_0010"].iloc[0]
print(row["heavy_paraphrase"])

The South Shore Line Airport Realignment refers to a proposed undertaking that would shift the South Shore Line’s primary terminus in Indiana from its present facility on the eastern side of South Bend International Airport to a newly built station west of the airport. The work would also involve laying new approach tracks. This redesign is expected to shorten travel times because the existing route into the airport station is indirect, slow, and includes numerous at-grade crossings.

The airport’s current station opened on November 20, 1992, and was selected for the east side because a freight spur already existed there. Nevertheless, that location was always regarded as a temporary arrangement for South Bend’s station. In 2006, officials started promoting the idea of transferring the facility to the airport’s western side, and interest in the proposal revived in 2017. Not long afterward, other possible station sites were proposed as well. Several alternatives for replacing the existi

Again, this is acceptable.

In [32]:
para_df[para_df["document_id"] == "BBC_0148"]

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
1437,BBC_0148,BBC,General,gpt-5.6-luna,Researchers from the Victor Chang Institute in...,Researchers from Sydney’s Victor Chang Institu...,Scientists at Sydney’s Victor Chang Institute ...,"Des chercheurs de l’Institut Victor Chang, à S...",Researchers at the Victor Chang Institute in S...,377,199,389,464,389


In [33]:
row = para_df[para_df["document_id"] == "BBC_0148"].iloc[0]
print(row["light_paraphrase"])

Researchers from Sydney’s Victor Chang Institute described it as “a double breakthrough”, after identifying both a cause and a preventative solution. With 7.9 million babies born worldwide each year with a birth defect, the team hopes the benefits will be far-reaching. However, one expert said the findings “cannot be translated into recommendations” for pregnancy.

The researchers examined the DNA of four families in which the mothers had experienced repeated miscarriages or their babies had been born with several birth defects, including heart, kidney, vertebrae and cleft palate problems. They identified mutations in two genes that left the child deficient in a vital molecule called Nicotinamide adenine dinucleotide (NAD), which enables cells to produce energy and organs to develop normally.

Lead researcher Prof Sally Dunwoodie reproduced these mutations in mice but discovered they could be corrected if the pregnant mother was given niacin (vitamin B3). “You can increase your NAD lev

This needs replacing. I will go through all now, given that two were acceptable.

In [34]:
para_df[para_df["document_id"] == "BBC_0344"]

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
975,BBC_0344,BBC,General,gpt-5.6-luna,Divers from the FBI began searching the waters...,FBI divers began combing the waters of Seccomb...,FBI divers were deployed to Seccombe Lake Park...,Des plongeurs du FBI ont commencé à fouiller l...,FBI divers have begun searching the waters of ...,364,271,360,427,371


In [35]:
row = para_df[para_df["document_id"] == "BBC_0344"].iloc[0]
print(row["light_paraphrase"])

FBI divers began combing the waters of Seccombe Lake Park, two miles (3km) from the location of the attack. The two suspects are believed to have visited the park on the day of the shooting, an FBI spokesman said. The attack is being treated as an act of terror, the deadliest in the US since 9/11. Tashfeen Malik and her husband, Syed Farook, opened fire at a workplace gathering for Farook's colleagues, killing 14 people and injuring 22.

Who were the attackers?

San Bernardino shooting: The victims

The couple, who officials say became radicalised several years ago, were both killed in a shoot-out with police hours after the attack. Speaking to reporters at the park on Thursday, the FBI refused to say exactly what the divers were searching for.

"We have indications through leads that at some point they came to this park," said David Bowdich, assistant director in charge of the FBI's Los Angeles field office.

US media, citing law enforcement sources, reported that investigators were l

In [36]:
para_df[para_df["document_id"] == "BBC_0080"]

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
738,BBC_0080,BBC,General,gpt-5.6-luna,Officer Jason Van Dyke is charged with the fir...,Officer Jason Van Dyke has been charged with t...,Chicago police officer Jason Van Dyke faces a ...,L’agent Jason Van Dyke est accusé du meurtre a...,Officer Jason Van Dyke is accused of the first...,377,88,109,468,383


In [37]:
row = para_df[para_df["document_id"] == "BBC_0080"].iloc[0]
print(row["light_paraphrase"])

Officer Jason Van Dyke has been charged with the first-degree murder of 17-year-old Laquan McDonald, who was stopped by police after slashing car tyres. In a Thanksgiving message on Facebook, the president praised Chicago protesters for maintaining the peace. Four people were arrested overnight as demonstrators marched through the city. The protests were neither as widespread nor as tense as those on Tuesday night, which started moments after police released dashcam footage of the incident. Mr Obama said on Facebook: “Like many Americans, I was deeply disturbed by


In [39]:
para_df[para_df["document_id"] == "RED_0111"]

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
690,RED_0111,Reddit,explainlikeimfive,gpt-5.6-luna,"This is my first answer to an ELI5, so stick w...","This is my first answer to an ELI5, so bear wi...",This is my first attempt at answering an ELI5 ...,"C’est ma première réponse à un ELI5, alors soy...","This is my first response to an ELI5, so pleas...",352,323,286,371,347


In [41]:
row = para_df[para_df["document_id"] == "RED_0111"].iloc[0]
print(row["heavy_paraphrase"])

This is my first attempt at answering an ELI5 question, so bear with me. Noise-cancelling headphones are a remarkably smart use of a fairly complex electronic system. I do not know enough about the circuitry to explain its technical details, so I will focus on the basic principle.

Each earcup contains a microphone. Together, these microphones listen to the sounds surrounding you, known as ambient noise, and send that information into the headphones’ electronic circuit. The sound is processed as a frequency-based signal, but that part is not essential to understand here. The important point is that sound travels through the air as a wave, much like waves moving across water.

Using the signal from the microphones, the circuit produces something called *antinoise*. This is designed to match the surrounding noise while being its precise opposite. If the ambient sound and the antinoise are played together correctly, they effectively eliminate one another, leaving no audible sound. This ha

This one is complete so keep.

In [42]:
para_df[para_df["document_id"] == "AMZ_0156"]

,document_id,source,topic,model,original_text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
430,AMZ_0156,Amazon reviews,Food reviews,gpt-5.6-luna,Dogswell jerky treat come China. They are not ...,Dogswell jerky treats come from China. They ar...,"Dogswell jerky treats are made in China, and t...",Les friandises séchées Dogswell viennent de Ch...,Dogswell dried treats come from China. They ar...,377,192,338,471,383


In [45]:
row = para_df[para_df["document_id"] == "AMZ_0156"].iloc[0]
print(row["light_paraphrase"])

Dogswell jerky treats come from China. They are not safe. I had a Subscribe & Save subscription for the product, but I cancelled it after reading this story from September 15, 2012. (Just search for the article’s title, which is the same as my review.) ---- The FDA now says that 350 dogs and one cat have died over the last 18 months, reportedly after eating chicken jerky treats made in China. An FDA report released this week says they have received 2,200 reports of illnesses connected to the treats—the deaths are included in that number. There is no geographic pattern to the illnesses and deaths—cases have been reported from all 50 states and 6 Canadian provinces. The FDA says most complaints involve chicken jerky (treats, tenders, and strips), but others include duck, sweet potato, and treats in which chicken or duck jerky is wrapped around dried fruits, sweet potatoes, or yams. The report goes on to state that over the past 10 years, there has been a dramatic rise in the importation 

Can now remove those that cut our mid-sentence and re-run the program to get them again then check. Note at this point that I did attempt to run through and look for texts that did not end in . ! ? etc. but ereh were hundreds and on inspection most of these were i.e. .) etc. so that wasn't helpful.

In [46]:
# list of ids of records above that cut out midway through paraphrase
incomplete_ids = [
    "BBC_0375",
    "RED_0127",
    "BBC_0309",
    "BBC_0148",
    "BBC_0344",
    "BBC_0080",
    "AMZ_0156"    
]

# delete these rows
para_df = para_df[~para_df["document_id"].isin(incomplete_ids)].reset_index(drop = True)

# dropping 78 from 2000, so expect 1993
print(len(para_df))

1993


We've now trimmed out the NaNs / duplicates, looked for those under 300 words and removed etc. so should now be able to run the main program again. Want to keep all the above text in order for reference so will copy cells down here again for completeness.

In [56]:
# but first need to save my results

para_df.drop(
    columns = [
        "original_word_count",
        "light_word_count",
        "heavy_word_count",
        "french_word_count",
        "translation_word_count",
    ], 
    inplace = True
)
        
para_df.to_csv("full_paraphrase_dataset_2.csv", index = False)

__________

### Re-run to get those last few that didn't complete.

All following cells copied from above.

In [60]:
csv_file = "full_paraphrase_dataset.csv"
df = pd.read_csv(csv_file)
main = df.copy()

In [61]:
# set new output file to contain all original text and info and new paraphrases
output_file = "full_paraphrase_dataset_2.csv"

# have now decided on luna model
model = "gpt-5.6-luna"


# if this does crash, I want to be able to re-run this cell which should check for existing
# data to determine where to carry on from, rather than re-doing old work
if os.path.exists(output_file):

    # read output file
    para_df = pd.read_csv(output_file)
    # convert df to list of dicts
    results = para_df.to_dict("records")
    # find number completed already
    print("Existing results found: ", len(results))

else:

    # if there were none existing this is first time therefore initialise list
    results = []
    print("Starting from beginning.")



# have established where we are startign from, now need to get the ids of any already complete
completed = {
    row["document_id"]
    for row in results
    if pd.notna(row.get("light_paraphrase"))
    and pd.notna(row.get("heavy_paraphrase"))
    and pd.notna(row.get("translation_paraphrase"))
}
print("Documents completed: ", len(completed))


############################################
# Begin looping through dataset
############################################

# now want to loop through whole dataset, initially skipping any rows that have
# a document ID within completed, established above
for i, (_, row) in enumerate(main.iterrows(), start = 1):

    # get id and text data from csv
    document_id = row["document_id"]
    original_text = row["original_text"]

    # now look for those ids in completed to see what can be skipped
    if document_id in completed:

        print(
            f"[{i}/{len(main)} ]"
            f"{document_id} already completed"
        )

        # if id existed move onto next and do nothing
        continue

    # otherwise we are now beginning to paraphrase those that havne't been done already
    print(f"\n====== Document {i}/{len(main)} ======")
    print("Document: ", document_id)

    
    ############################################
    # Now we move onto the paraphrasing requests
    ############################################

    try:

        # light paraphrase
        print("Light paraphrase...")
        # add original text from df into light paraphrase prompt
        light = send_prompt(
            light_para_prompt.format(text = original_text),
            model
        )

        # Heavy paraphrase
        print("Heavy paraphrase...")
        # add original text from df into heavy paraphrase prompt
        heavy = send_prompt(
            heavy_para_prompt.format(text = original_text),
            model
        )

        # eng to fre translation
        print("Eng > Fre translation...")
        # add original text from df into translation prompt
        french = send_prompt(
            trans_french_prompt.format(text = original_text),
            model
        )

        # fre to eng translation
        print("Fre > Eng translation...")
        # add french text into translation prompt
        english = send_prompt(
            trans_english_prompt.format(text = french),
            model
        )

        # store output
        results.append({
            "document_id": document_id,
            "source": row["source"],
            "topic": row["topic"],
            "model": model,
            "original_text": original_text,
            "light_paraphrase": light,
            "heavy_paraphrase": heavy,
            "french_translation": french,
            "translation_paraphrase": english
        })

        # also add doc id to completed
        completed.add(document_id)
        print("Document complete")


    ############################################
    # Handle API request errors
    ############################################

    # want to state what the error was and still add the row to the document 
    # so that it is logged and we can continue
    except Exception as e:

        print("Error:", e)

        # still want to store output to find issue later if necessary
        results.append({
            "document_id": document_id,
            "source": row["source"],
            "topic": row["topic"],
            "model": model,
            "original_text": original_text,             
            "light_paraphrase": None,
            "heavy_paraphrase": None,
            "french_translation": None,
            "translation_paraphrase": None
        })

        print("Failed: ", document_id)


    ############################################
    # Save
    ############################################

    # want to save to the csv after each document rather than waiting 
    # until very end in case it crashes
    para_df = pd.DataFrame(results)
    para_df.to_csv(output_file, index = False)
    print("Results saved")
    # small half second pause between models to avoid overuse error
    time.sleep(0.5)

Existing results found:  1993
Documents completed:  1993
[1/2003 ]WIKI_0122 already completed
[2/2003 ]BBC_0281 already completed
[3/2003 ]WIKI_0313 already completed
[4/2003 ]AMZ_0214 already completed
[5/2003 ]BBC_0278 already completed
[6/2003 ]WIKI_0204 already completed
[7/2003 ]WIKI_0359 already completed
[8/2003 ]GUT_0281 already completed
[9/2003 ]RED_0322 already completed
[10/2003 ]BBC_0192 already completed
[11/2003 ]BBC_0077 already completed
[12/2003 ]WIKI_0353 already completed
[13/2003 ]RED_0142 already completed
[14/2003 ]GUT_0078 already completed
[15/2003 ]GUT_0191 already completed
[16/2003 ]AMZ_0089 already completed
[17/2003 ]RED_0084 already completed
[18/2003 ]RED_0228 already completed
[19/2003 ]GUT_0176 already completed
[20/2003 ]AMZ_0165 already completed
[21/2003 ]GUT_0116 already completed
[22/2003 ]BBC_0320 already completed
[23/2003 ]WIKI_0389 already completed
[24/2003 ]WIKI_0012 already completed
[25/2003 ]GUT_0220 already completed
[26/2003 ]GUT_0395 a

This appears to be complete - can now re-run the same checks from earlier to establish that the dataset is complete.

In [62]:
len(para_df)

2000

In [63]:
para_df["document_id"].duplicated().sum()

0

In [64]:
para_df["document_id"].duplicated().sum()

0

In [66]:
para_df.isna().sum()

document_id               0
source                    0
topic                     0
model                     0
original_text             0
light_paraphrase          0
heavy_paraphrase          0
french_translation        0
translation_paraphrase    0
dtype: int64

In [67]:
# split string in each cell in the original text row to words to calculate
# word count, do for every row in column...
para_df["original_word_count"] = (
    para_df["original_text"]
    .fillna("")
    .str.split()
    .str.len()
)

# and do for all 5 text columns...
para_df["light_word_count"] = (
    para_df["light_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

para_df["heavy_word_count"] = (
    para_df["heavy_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

para_df["french_word_count"] = (
    para_df["french_translation"]
    .fillna("")
    .str.split()
    .str.len()
)
para_df["translation_word_count"] = (
    para_df["translation_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

print(para_df[
    [
        "original_word_count",
        "light_word_count",
        "heavy_word_count",
        "french_word_count",
        "translation_word_count"
    ]
].describe())

       original_word_count  light_word_count  heavy_word_count  \
count          2000.000000       2000.000000       2000.000000   
mean            398.800000        390.209500        404.746000   
std              23.693748         27.175869         35.780212   
min             311.000000        101.000000        112.000000   
25%             381.000000        374.000000        381.000000   
50%             404.000000        395.000000        407.000000   
75%             414.000000        408.000000        428.000000   
max             451.000000        455.000000        525.000000   

       french_word_count  translation_word_count  
count        2000.000000             2000.000000  
mean          458.611500              408.177500  
std            38.416005               28.303861  
min           343.000000              305.000000  
25%           432.000000              391.000000  
50%           458.000000              410.000000  
75%           485.000000              426.000000

It still appears we have some < 300 texts...

RED_0111, WIKI_0010, AMZ_0394 were previously realsied to be small original texts where paraphrasing changed the word count to under 300 and they were therefore acceptable, however the rest appear to have issues. However, on inspection it appears that all these are the light paraphrase only, so i will re-prompt and try one final time.

In [78]:
# prompt copied from above and expanded

light_para_prompt = """

Paraphrase the following text lightly.

YOU MUST PARAPHRASE THE ENTIRE TEXT FROM START TO FINISH.

Make limited changes to the original wording, phrasing and sentence structure.
Replace some words and phrases with alternatives.
Make minor grammatical changes.

Preserve the original meaning, factual information, level of detail, structure and ideas as closely as possible.
The result should be a paraphrase that remains very close to the original text.
Do not add new information, remove information, or summarise the text.

Keep the length similar to the original. The original is 350-450 words so return a paraphrase from 350-450 words.
Continue paraphrasing until you reach the end of the original text.
Do not stop midway through the source.
The paraphrase must end with a complete sentence.

Return only the paraphrased text.


TEXT: {text}

"""

In [79]:
failed_ids = [
    "BBC_0375",	
    "BBC_0309",
    "AMZ_0156",
    "BBC_0080",
    "BBC_0344"
]

In [81]:
for document_id in failed_ids:

    print("Re-prmopting: ", document_id)
    # find row
    row_index = para_df.index[para_df["document_id"] == document_id][0]
    # get original text
    original_text = para_df.loc[row_index, "original_text"]

    # put original text through new light prompt
    new_light_para = send_prompt(
        light_para_prompt.format(
        text = original_text
        ),
        "gpt-5.6-luna"
    )

    # get new word count to be sure
    new_word_count = len(new_light_para.split())
    print("Word count: ", new_word_count)

    # replace the light paraphrase
    para_df.loc[row_index, "light_paraphrase"] = new_light_para
    print("Paraphrase replaced.")
    

Re-prmopting:  BBC_0375
Word count:  374
Paraphrase replaced.
Re-prmopting:  BBC_0309
Word count:  430
Paraphrase replaced.
Re-prmopting:  AMZ_0156
Word count:  256
Paraphrase replaced.
Re-prmopting:  BBC_0080
Word count:  103
Paraphrase replaced.
Re-prmopting:  BBC_0344
Word count:  282
Paraphrase replaced.


Still happening! Letrs inspect.

In [84]:
para_df[para_df["document_id"] == "AMZ_0156"]["light_paraphrase"].iloc[0]

'Dogswell jerky treats come from China. They are not safe. I had a subscribe-and-save subscription for this product, but I cancelled it after reading this story from September 15, 2012. (Just search for the article’s title, which is also the title of my review.) ---- The FDA now reports that 350 dogs and one cat have died during the past 18 months, reportedly after consuming chicken jerky treats manufactured in China. An FDA report released this week states that the agency has received 2,200 reports of illnesses associated with these treats, with the deaths included in that total. There is no particular geographic pattern connected to the illnesses and deaths; cases have been reported in all 50 states and six Canadian provinces. The FDA says most complaints concern chicken jerky products, including treats, tenders, and strips, although other products involve duck, sweet potato, and treats in which chicken or duck jerky is wrapped around dried fruit, sweet potatoes, or yams.\n\nThe repo

In [85]:
para_df[para_df["document_id"] == "BBC_0080"]["light_paraphrase"].iloc[0]

'Officer Jason Van Dyke faces a first-degree murder charge over the shooting of 17-year-old Laquan McDonald, who was stopped by police after reportedly slashing car tyres. In a Thanksgiving post on Facebook, the president praised Chicago protesters for maintaining peace. Four people were arrested overnight while demonstrators marched through the city. The protests were neither as extensive nor as tense as those on Tuesday night, which began shortly after police released dashcam footage of the incident.\n\nMr Obama wrote on Facebook: “Like many Americans, I was deeply disturbed by the footage of the fatal shooting of 17-year-old Laquan McDonald.\n\n“This Thanksgiving, I ask'

In [86]:
para_df[para_df["document_id"] == "BBC_0344"]["light_paraphrase"].iloc[0]

'FBI divers began combing the waters of Seccombe Lake Park, located two miles (3km) from the scene of the attack. An FBI spokesman said the two suspects are believed to have visited the park on the day of the shooting. The incident is being treated as an act of terrorism and is the deadliest attack in the US since 9/11. Tashfeen Malik and her husband, Syed Farook, opened fire during a workplace gathering for Farook’s colleagues, killing 14 people and injuring 22. Who were the attackers? San Bernardino shooting: The victims\n\nOfficials say the couple had become radicalised several years ago. Both were killed in a gun battle with police several hours after the attack. Speaking to reporters at the park on Thursday, the FBI refused to say exactly what the divers were searching for. “We have indications through leads that at some point they came to this park,” said David Bowdich, the FBI assistant director in charge of its Los Angeles field office.\n\nUS media, citing law enforcement sourc

\n tabs were cleared out initially in pre-processing so something is going wrong withthe prmpting to re-introduce them. They will be re-cleaned later again so that isn't a worry and I will keep BBC_0344 as it ends with the end of a sentence at almost 300 words but will drop AMZ-0156 and BBC_0080 entirely now.

In [87]:
remove_ids = [
    "AMZ_0156",
    "BBC_0080"
]

para_df = para_df[~para_df["document_id"].isin(remove_ids)].reset_index(drop = True)    

In [88]:
len(para_df)

1998

In [90]:
para_df.drop(
    columns = [
        "original_word_count",
        "light_word_count",
        "heavy_word_count",
        "french_word_count",
        "translation_word_count",
    ], 
    inplace = True
)

In [91]:
para_df.to_csv("full_paraphrase_dataset_2.csv", index = False)